# Predictive Maintenance Inference - Batch or serverless real-time


With AutoML, our best model was automatically saved in our MLFlow registry.

All we need to do now is use this model to run Inferences. A simple solution is to share the model name to our Data Engineering team and they'll be able to call this model within the pipeline they maintained. That's what we did in our Spark Declarative Pipelines pipeline!

Alternatively, this can be schedule in a separate job. Here is an example to show you how MLFlow can be directly used to retriver the model and run inferences.

*Make sure you run the previous notebook to be able to access the data.*

## Environment Recreation
The cell below downloads the model artifacts associated with your model in the remote registry, which include `conda.yaml` and `requirements.txt` files. In this notebook, `pip` is used to reinstall dependencies by default.

<!-- Collect usage data (view). Remove it to disable collection. View README for more details.  -->
<img width="1px" src="https://ppxrzfxige.execute-api.us-west-2.amazonaws.com/v1/analytics?category=lakehouse&org_id=2162748966026566&notebook=%2F04-Data-Science-ML%2F04.3-running-inference-iot-turbine&demo_name=lakehouse-iot-platform&event=VIEW&path=%2F_dbdemos%2Flakehouse%2Flakehouse-iot-platform%2F04-Data-Science-ML%2F04.3-running-inference-iot-turbine&version=1">

In [0]:
%pip install mlflow==3.1.1 databricks-sdk==0.59.0

  Using cached mlflow-3.1.1-py3-none-any.whl.metadata (29 kB)
  Using cached databricks_sdk-0.59.0-py3-none-any.whl.metadata (39 kB)
  Using cached mlflow_skinny-3.1.1-py3-none-any.whl.metadata (30 kB)
  Using cached docker-7.1.0-py3-none-any.whl.metadata (3.8 kB)
  Using cached graphene-3.4.3-py2.py3-none-any.whl.metadata (6.9 kB)
  Using cached graphql_relay-3.2.0-py3-none-any.whl.metadata (12 kB)
Using cached mlflow-3.1.1-py3-none-any.whl (24.7 MB)
Using cached databricks_sdk-0.59.0-py3-none-any.whl (676 kB)
Using cached mlflow_skinny-3.1.1-py3-none-any.whl (1.9 MB)
Using cached docker-7.1.0-py3-none-any.whl (147 kB)
Using cached graphene-3.4.3-py2.py3-none-any.whl (114 kB)
Using cached graphql_relay-3.2.0-py3-none-any.whl (16 kB)
  Attempting uninstall: databricks-sdk
    Found existing installation: databricks-sdk 0.30.0
    Not uninstalling databricks-sdk at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-5181abc2-2

In [0]:
%run ../_resources/00-setup $reset_all_data=false

USE CATALOG `main_build`
using catalog.database `main_build`.`dbdemos_iot_platform`


data already existing. Run with reset_all_data=true to force a data cleanup for your local demo.


##Deploying the model for batch inferences

<img style="float: right; margin-left: 20px" width="800" src="https://github.com/databricks-demos/dbdemos-resources/blob/main/images/retail/lakehouse-churn/ep_model_serving_creation.gif?raw=true" />

Now that our model is available in the Registry, we can load it to compute our inferences and save them in a table to start building dashboards.

We will use MLFlow function to load a pyspark UDF and distribute our inference in the entire cluster. If the data is small, we can also load the model with plain python and use a pandas Dataframe.

If you don't know how to start, Databricks can generate a batch inference notebook in just one click from the model registry: Open MLFlow model registry and click the "User model for inference" button!

### Scaling inferences using Spark 
We'll first see how it can be loaded as a spark UDF and called directly in a SQL function:

In [0]:
import mlflow
mlflow.set_registry_uri('databricks-uc')
#                                                                                                    Stage/version  
#                                                                                       Model name         |        
#                                                                                           |              |        
predict_maintenance = mlflow.pyfunc.spark_udf(spark, f"models:/{catalog}.{db}.dbdemos_turbine_maintenance@prod", "string", env_manager='virtualenv')
#We can use the function in SQL
spark.udf.register("predict_maintenance", predict_maintenance)
columns = predict_maintenance.metadata.get_input_schema().input_names()

2025/10/27 16:34:02 INFO mlflow.pyfunc: This UDF will use virtualenv to recreate the model's software environment for inference. This may take extra time during execution.


2025/10/27 16:34:02 INFO mlflow.models.flavor_backend_registry: Selected backend for flavor 'python_function'
2025/10/27 16:34:02 INFO mlflow.utils.virtualenv: Installing python 3.12.3 if it does not exist
2025/10/27 16:35:14 INFO mlflow.utils.virtualenv: Creating a new environment in /local_disk0/.ephemeral_nfs/repl_tmp_data/ReplId-19a26-7df7b-b/mlflow/envs/virtualenv_envs/mlflow-836b50d3da336556c48f20a130dfb71030985159 with /local_disk0/.ephemeral_nfs/repl_tmp_data/ReplId-19a26-7df7b-b/mlflow/envs/pyenv_root/versions/3.12.3/bin/python
2025/10/27 16:35:14 INFO mlflow.utils.virtualenv: Installing dependencies
2025/10/27 16:36:02 INFO mlflow.utils.environment: === Running command '['bash', '-c', 'source /local_disk0/.ephemeral_nfs/repl_tmp_data/ReplId-19a26-7df7b-b/mlflow/envs/virtualenv_envs/mlflow-836b50d3da336556c48f20a130dfb71030985159/bin/activate && python -c ""']'


In [0]:
columns = predict_maintenance.metadata.get_input_schema().input_names()
spark.table('turbine_hourly_features').withColumn("dbdemos_turbine_maintenance", predict_maintenance(*columns)).display()

turbine_id,hourly_timestamp,avg_energy,std_sensor_A,std_sensor_B,std_sensor_C,std_sensor_D,std_sensor_E,std_sensor_F,location,model,state,abnormal_sensor,dbdemos_turbine_maintenance
004a641f-e9e5-9fff-d421-1bf88319420b,2024-01-16T17:00:00Z,0.18897920400916968,0.9644652043128557,2.6558386572409107,3.452810601357622,2.4851587526074033,2.288403246836928,4.702138990110718,Tupelo,EpicWind,America/Chicago,sensor_F,ok
004a641f-e9e5-9fff-d421-1bf88319420b,2024-01-16T18:00:00Z,0.19212257629921778,1.0681855556261903,2.3848184303882842,3.303412042721335,2.172251292324,2.3425930195968947,4.870875418724549,Tupelo,EpicWind,America/Chicago,sensor_F,ok
004a641f-e9e5-9fff-d421-1bf88319420b,2024-01-16T19:00:00Z,0.17356344574506763,1.14208877201463,2.062708699095104,3.0193296637120044,2.3395520448680496,2.7306978700770164,4.237196637787602,Tupelo,EpicWind,America/Chicago,sensor_F,ok
004a641f-e9e5-9fff-d421-1bf88319420b,2024-01-16T20:00:00Z,0.10343409262714737,1.0498727154061809,2.2192165091594975,3.2467261389316113,2.3204665834317817,2.6627001776134547,4.289404582190178,Tupelo,EpicWind,America/Chicago,sensor_F,ok
004a641f-e9e5-9fff-d421-1bf88319420b,2024-01-16T21:00:00Z,0.15481243527493332,1.0325552090494654,2.142101655549622,2.729842321266221,2.3597486817214515,2.7614663980581695,4.588788770497015,Tupelo,EpicWind,America/Chicago,sensor_F,ok
004a641f-e9e5-9fff-d421-1bf88319420b,2024-01-16T22:00:00Z,0.08477232550242081,1.0021697211227556,2.096894376529207,2.9215472587753384,2.477840322666964,2.9466029618007323,4.357159925464822,Tupelo,EpicWind,America/Chicago,sensor_F,ok
004a641f-e9e5-9fff-d421-1bf88319420b,2024-01-16T23:00:00Z,0.07481860903879099,1.058048335093487,2.485293271624966,2.892716085289315,2.156705095595585,2.21203585297937,5.6145260271394255,Tupelo,EpicWind,America/Chicago,sensor_F,ok
00f27248-1f4f-e174-432c-53bd2a9158df,2024-01-16T17:00:00Z,0.12839653721057287,1.0656088831997512,1.926331925310217,3.3330563526547743,2.23004019614146,2.3546260863866486,1.891304903160798,Crystal Lake,EpicWind,America/Chicago,ok,ok
00f27248-1f4f-e174-432c-53bd2a9158df,2024-01-16T18:00:00Z,0.85422454913039,1.0803097778159463,1.9618452098136365,2.971742610514548,2.306627597988137,2.516697368859581,1.980452948870913,Crystal Lake,EpicWind,America/Chicago,ok,ok
00f27248-1f4f-e174-432c-53bd2a9158df,2024-01-16T19:00:00Z,0.49155356663955985,1.0646332592567709,2.2186746553400307,3.345943840796343,2.2847856939507163,2.5560343320959498,1.9519204325253463,Crystal Lake,EpicWind,America/Chicago,ok,ok


In [0]:
%sql
SELECT turbine_id, predict_maintenance(hourly_timestamp, avg_energy, std_sensor_A, std_sensor_B, std_sensor_C, std_sensor_D, std_sensor_E, std_sensor_F, location, model, state) as prediction FROM turbine_hourly_features

turbine_id,prediction
004a641f-e9e5-9fff-d421-1bf88319420b,ok
004a641f-e9e5-9fff-d421-1bf88319420b,ok
004a641f-e9e5-9fff-d421-1bf88319420b,ok
004a641f-e9e5-9fff-d421-1bf88319420b,ok
004a641f-e9e5-9fff-d421-1bf88319420b,ok
004a641f-e9e5-9fff-d421-1bf88319420b,ok
004a641f-e9e5-9fff-d421-1bf88319420b,ok
00f27248-1f4f-e174-432c-53bd2a9158df,ok
00f27248-1f4f-e174-432c-53bd2a9158df,ok
00f27248-1f4f-e174-432c-53bd2a9158df,ok


### Pure pandas inference
If we have a small dataset, we can also compute our segment using a single node and pandas API:

In [0]:
from mlflow.store.artifact.models_artifact_repo import ModelsArtifactRepository
mlflow.set_registry_uri('databricks-uc')
local_path = ModelsArtifactRepository(f"models:/{catalog}.{db}.dbdemos_turbine_maintenance@prod").download_artifacts("") # download model from remote registry

requirements_path = os.path.join(local_path, "requirements.txt")
if not os.path.exists(requirements_path):
  dbutils.fs.put("file:" + requirements_path, "", True)

In [0]:
%pip install -r $requirements_path
dbutils.library.restartPython()

  Using cached pandas-2.2.3-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (89 kB)
  Using cached tzdata-2025.2-py2.py3-none-any.whl.metadata (1.4 kB)
Using cached pandas-2.2.3-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (12.7 MB)
Using cached tzdata-2025.2-py2.py3-none-any.whl (347 kB)
  Attempting uninstall: pandas
    Found existing installation: pandas 1.5.3
    Not uninstalling pandas at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-5181abc2-261f-469d-804d-ff3380515027
    Can't uninstall 'pandas'. No files were found to uninstall.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
%run ../_resources/00-setup $reset_all_data=false

USE CATALOG `main_build`
using catalog.database `main_build`.`dbdemos_iot_platform`


data already existing. Run with reset_all_data=true to force a data cleanup for your local demo.


In [0]:
import mlflow
mlflow.set_registry_uri('databricks-uc')
model = mlflow.pyfunc.load_model(f"models:/{catalog}.{db}.dbdemos_turbine_maintenance@prod")
columns = model.metadata.get_input_schema().input_names()
df = spark.table('turbine_hourly_features').select(*columns).limit(10).toPandas()
df['churn_prediction'] = model.predict(df)
display(df)

[LightGBM] [Warning] lambda_l2 is set=27.761059765184243, reg_lambda=0.0 will be ignored. Current value: lambda_l2=27.761059765184243
[LightGBM] [Warning] lambda_l1 is set=40.596326947371445, reg_alpha=0.0 will be ignored. Current value: lambda_l1=40.596326947371445


hourly_timestamp,avg_energy,std_sensor_A,std_sensor_B,std_sensor_C,std_sensor_D,std_sensor_E,std_sensor_F,location,model,state,churn_prediction
2024-01-16T17:00:00Z,0.18897920400916968,0.9644652043128557,2.6558386572409107,3.452810601357622,2.4851587526074033,2.288403246836928,4.702138990110718,Tupelo,EpicWind,America/Chicago,ok
2024-01-16T18:00:00Z,0.19212257629921778,1.0681855556261903,2.3848184303882842,3.303412042721335,2.172251292324,2.3425930195968947,4.870875418724549,Tupelo,EpicWind,America/Chicago,ok
2024-01-16T19:00:00Z,0.17356344574506763,1.14208877201463,2.062708699095104,3.0193296637120044,2.3395520448680496,2.7306978700770164,4.237196637787602,Tupelo,EpicWind,America/Chicago,ok
2024-01-16T20:00:00Z,0.10343409262714737,1.0498727154061809,2.2192165091594975,3.2467261389316113,2.3204665834317817,2.6627001776134547,4.289404582190178,Tupelo,EpicWind,America/Chicago,ok
2024-01-16T21:00:00Z,0.15481243527493332,1.0325552090494654,2.142101655549622,2.729842321266221,2.3597486817214515,2.7614663980581695,4.588788770497015,Tupelo,EpicWind,America/Chicago,ok
2024-01-16T22:00:00Z,0.08477232550242081,1.0021697211227556,2.096894376529207,2.9215472587753384,2.477840322666964,2.9466029618007323,4.357159925464822,Tupelo,EpicWind,America/Chicago,ok
2024-01-16T23:00:00Z,0.07481860903879099,1.058048335093487,2.485293271624966,2.892716085289315,2.156705095595585,2.21203585297937,5.6145260271394255,Tupelo,EpicWind,America/Chicago,ok
2024-01-16T17:00:00Z,0.12839653721057287,1.0656088831997512,1.926331925310217,3.3330563526547743,2.23004019614146,2.3546260863866486,1.891304903160798,Crystal Lake,EpicWind,America/Chicago,ok
2024-01-16T18:00:00Z,0.85422454913039,1.0803097778159463,1.9618452098136365,2.971742610514548,2.306627597988137,2.516697368859581,1.980452948870913,Crystal Lake,EpicWind,America/Chicago,ok
2024-01-16T19:00:00Z,0.49155356663955985,1.0646332592567709,2.2186746553400307,3.345943840796343,2.2847856939507163,2.5560343320959498,1.9519204325253463,Crystal Lake,EpicWind,America/Chicago,ok



## Realtime model serving with Databricks serverless serving

<img style="float: right; margin-left: 20px" width="800" src="https://github.com/databricks-demos/dbdemos-resources/blob/main/images/retail/lakehouse-churn/ep_model_serving_creation.gif?raw=true" />

Databricks also provides serverless serving.

Click on model Serving, enable realtime serverless and your endpoint will be created, providing serving over REST api within a Click.

Databricks Serverless offer autoscaling, including downscaling to zero when you don't have any traffic to offer best-in-class TCO while keeping low-latencies model serving.

## Real time model inference

Let's now deploy our model behind a realtime model serving endpoint.

We'll then use this endpoint in our GenAI Agentic demo to be able to fetch a turbine status in realtime


In [0]:
from mlflow.deployments import get_deploy_client

client = get_deploy_client("databricks")
try:
    endpoint = client.create_endpoint(
        name=MODEL_SERVING_ENDPOINT_NAME,
        config={
            "served_entities": [
                {
                    "name": "iot-maintenance-serving-endpoint",
                    "entity_name": f"{catalog}.{db}.{model_name}",
                    "entity_version": get_last_model_version(f"{catalog}.{db}.{model_name}"),
                    "workload_size": "Small",
                    "scale_to_zero_enabled": True
                }
            ]
        }
    )
except Exception as e:
    if "already exists" in str(e):
        print(f"Endpoint {catalog}.{db}.{MODEL_SERVING_ENDPOINT_NAME} already exists. Skipping creation.")
    else:
        raise e

while client.get_endpoint(MODEL_SERVING_ENDPOINT_NAME)['state']['config_update'] == 'IN_PROGRESS':
    time.sleep(10)

Endpoint main.dbdemos_iot_platform.dbdemos_iot_turbine_prediction_endpoint already exists. Skipping creation.


You can now view the status of the Feature Serving Endpoint in the table on the **Serving endpoints** page. Click **Serving** in the sidebar to display the page.

In [0]:
from mlflow import deployments

def score_model(dataset):
  client = mlflow.deployments.get_deploy_client("databricks")
  predictions = client.predict(endpoint=MODEL_SERVING_ENDPOINT_NAME, inputs=dataset.to_dict(orient='split'))

dataset = spark.table(f'turbine_hourly_features').select(*columns).limit(3).toPandas()
#Deploy your model and uncomment to run your inferences live!
#score_model(dataset)

Another way to run inference is the use of ai_query. ai_query is one of multiple Databricks SQL ai function that allows you to invoke a machine learning model serving endpoint directly from SQL queries.
It sends structured input data to the specified model endpoint and returns predictions as part of your query results.
Benefits:
- Enables seamless integration of ML predictions into SQL analytics workflows.
- Allows batch scoring on large datasets using SQL.
- Simplifies operationalization of ML models for business users and analysts.
- Reduces the need for custom Python code to call model endpoints.

Find out more here: https://docs.databricks.com/aws/en/large-language-models/ai-functions

Example usage in Databricks SQL:
```
SELECT *, ai_query(
        'model_serving_endpoint_name',
        named_struct(
            'feature1', feature1,
            'feature2', feature2,
            ...
        ),
        'STRING'
    ) as prediction
FROM your_table
```

In [0]:
%sql
SELECT *, ai_query(
        'dbdemos_iot_turbine_prediction_endpoint', 
        named_struct(
            'hourly_timestamp', hourly_timestamp,
            'avg_energy', avg_energy,
            'std_sensor_A', std_sensor_A,
            'std_sensor_B', std_sensor_B,
            'std_sensor_C', std_sensor_C,
            'std_sensor_D', std_sensor_D,
            'std_sensor_E', std_sensor_E,
            'std_sensor_F', std_sensor_F,
            'location', location,
            'model', model,
            'state', state),
        'STRING'
    ) as prediction
FROM turbine_hourly_features

turbine_id,hourly_timestamp,avg_energy,std_sensor_A,std_sensor_B,std_sensor_C,std_sensor_D,std_sensor_E,std_sensor_F,location,model,state,abnormal_sensor,prediction
004a641f-e9e5-9fff-d421-1bf88319420b,2024-01-16T17:00:00Z,0.18897920400916968,0.9644652043128557,2.6558386572409107,3.452810601357622,2.4851587526074033,2.288403246836928,4.702138990110718,Tupelo,EpicWind,America/Chicago,sensor_F,ok
004a641f-e9e5-9fff-d421-1bf88319420b,2024-01-16T18:00:00Z,0.19212257629921778,1.0681855556261903,2.3848184303882842,3.303412042721335,2.172251292324,2.3425930195968947,4.870875418724549,Tupelo,EpicWind,America/Chicago,sensor_F,ok
004a641f-e9e5-9fff-d421-1bf88319420b,2024-01-16T19:00:00Z,0.17356344574506763,1.14208877201463,2.062708699095104,3.0193296637120044,2.3395520448680496,2.7306978700770164,4.237196637787602,Tupelo,EpicWind,America/Chicago,sensor_F,ok
004a641f-e9e5-9fff-d421-1bf88319420b,2024-01-16T20:00:00Z,0.10343409262714737,1.0498727154061809,2.2192165091594975,3.2467261389316113,2.3204665834317817,2.6627001776134547,4.289404582190178,Tupelo,EpicWind,America/Chicago,sensor_F,ok
004a641f-e9e5-9fff-d421-1bf88319420b,2024-01-16T21:00:00Z,0.15481243527493332,1.0325552090494654,2.142101655549622,2.729842321266221,2.3597486817214515,2.7614663980581695,4.588788770497015,Tupelo,EpicWind,America/Chicago,sensor_F,ok
004a641f-e9e5-9fff-d421-1bf88319420b,2024-01-16T22:00:00Z,0.08477232550242081,1.0021697211227556,2.096894376529207,2.9215472587753384,2.477840322666964,2.9466029618007323,4.357159925464822,Tupelo,EpicWind,America/Chicago,sensor_F,ok
004a641f-e9e5-9fff-d421-1bf88319420b,2024-01-16T23:00:00Z,0.07481860903879099,1.058048335093487,2.485293271624966,2.892716085289315,2.156705095595585,2.21203585297937,5.6145260271394255,Tupelo,EpicWind,America/Chicago,sensor_F,ok
00f27248-1f4f-e174-432c-53bd2a9158df,2024-01-16T17:00:00Z,0.12839653721057287,1.0656088831997512,1.926331925310217,3.3330563526547743,2.23004019614146,2.3546260863866486,1.891304903160798,Crystal Lake,EpicWind,America/Chicago,ok,ok
00f27248-1f4f-e174-432c-53bd2a9158df,2024-01-16T18:00:00Z,0.85422454913039,1.0803097778159463,1.9618452098136365,2.971742610514548,2.306627597988137,2.516697368859581,1.980452948870913,Crystal Lake,EpicWind,America/Chicago,ok,ok
00f27248-1f4f-e174-432c-53bd2a9158df,2024-01-16T19:00:00Z,0.49155356663955985,1.0646332592567709,2.2186746553400307,3.345943840796343,2.2847856939507163,2.5560343320959498,1.9519204325253463,Crystal Lake,EpicWind,America/Chicago,ok,ok



# Next step: Leverage inferences and automate action to lower cost

## Automate action to react on potential turbine failure

We now have an end 2 end data pipeline analizing and predicting churn. We can now easily trigger actions to reduce downtime, such as dispatching a team earlier to fix the issue before an actual outage!

## Track windturbine failure and impact

Of course, this prediction can be re-used in our dashboard to analyse future failure and measure impact. 

<img width="800px" src="https://github.com/databricks-demos/dbdemos-resources/raw/main/images/manufacturing/lakehouse-iot-turbine/lakehouse-manuf-iot-dashboard-2.png">

<a dbdemos-dashboard-id="turbine-predictive" href="/sql/dashboardsv3/01f1b2d43a181290aa14c87b87e2c449">Open the Predictive Maintenance AI/BI dashboard</a> | [Go back to the introduction]($../00-IOT-wind-turbine-introduction-DI-platform)